# 05. Trading strategy backtest

**Question.** Does a simple rule that buys the less popular team at tipoff make money after Kalshi's trading fees?

**Plan.** This notebook follows `ANALYSIS_PLAN.md` section 9.3, as amended and committed before any result here was computed. There is one rule, fixed in the plan. Nothing is tuned.

**Data.** `data/processed/snapshots.csv`, main sample at horizon 0, and `data/processed/team_metro_population.csv`.

**Seed.** The bootstrap uses seed 4981 and 2,000 replicates, as in the plan.

In [1]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
from IPython.display import display

from csv_io import write_csv
from metro_population import load_metro_population
from stats_tools import B, SEED
from strategy import (BURN_IN_GAMES, FEE_CHECKED, FEE_SOURCES, MIN_PRIOR_GAMES, RESULTS_DIR,
                      STAKE, THRESHOLD_QUANTILE, load_games, run_strategy, sample_counts,
                      taker_fee)

RESULTS_DIR.mkdir(exist_ok=True)
print(f"Bootstrap: B = {B:,}, seed = {SEED}")
print(f"Rule: {MIN_PRIOR_GAMES} earlier games per team, first {BURN_IN_GAMES} eligible games "
      f"set the threshold, quantile {THRESHOLD_QUANTILE}, stake ${STAKE:.0f} per trade")

Bootstrap: B = 2,000, seed = 4981
Rule: 10 earlier games per team, first 50 eligible games set the threshold, quantile 0.75, stake $100 per trade


## 1. The rule

Everything the rule uses is known at tipoff.

1. **Popularity.** For each team, the average pregame volume on its own market over the games it has already played. A game is eligible when both teams have at least 10 earlier games.
2. **Gap.** log(home team's average) minus log(away team's average).
3. **Threshold.** The 75th percentile of the absolute gap over the eligible games played before this one. The first 50 eligible games only set the threshold. No trade is made in them.
4. **Trade.** In a later eligible game whose absolute gap is at or above the threshold, stake $100 on the less popular team at its ask price at tipoff. Contracts = floor(100 / ask).
5. **Payout.** Each contract pays $1 if the team wins and nothing if it loses.

## 2. Fees

Every order pays Kalshi's taker fee once: 0.07 · C · P · (1 − P), rounded up to the next cent, where C is the number of contracts and P the ask price in dollars. There is no settlement fee.

In [2]:
print(f"Fee sources, checked on {FEE_CHECKED}:")
for what, url in FEE_SOURCES:
    print(f"- {what}\n  {url}")
examples = pd.DataFrame([{"contracts": c, "ask": p, "fee": taker_fee(c, p)[0],
                          "fee_unrounded": taker_fee(c, p)[1]}
                         for c, p in [(100, 0.05), (100, 0.50), (400, 0.25), (133, 0.75)]])
display(examples)

Fee sources, checked on 2026-11-23:
- Kalshi fee schedule as filed with the CFTC: fees = round up(0.07 x C x P x (1-P)), rounded to the next cent; no settlement fee
  https://cftc.gov/filings/orgrules/rule091222kexdcm003.pdf
- Kalshi API, GET /series/KXNBAGAME: fee_type = quadratic_with_maker_fees, fee_multiplier = 1; GET /series/fee_changes lists no change for the series
  https://api.elections.kalshi.com/trade-api/v2/series/KXNBAGAME


,contracts,ask,fee,fee_unrounded
0,100,0.05,0.34,0.332500
1,100,0.50,1.75,1.750000
2,400,0.25,5.25,5.250000
3,133,0.75,1.75,1.745625


The fee is charged once per order, and only the total is rounded. 100 contracts at $0.05 give 0.3325, which is charged as $0.34. An order whose fee is already a whole number of cents is not rounded up: 400 contracts at $0.25 pay exactly $5.25.

For a $100 stake the fee is close to 7 · (1 − P) dollars, so it is larger for cheap contracts: about $6.65 at a price of 0.05, $3.50 at 0.50 and $0.35 at 0.95.

## 3. Games the rule can trade

In [3]:
metro = load_metro_population()
games = load_games(metro)
counts = sample_counts(games)
write_csv(counts, RESULTS_DIR / "strategy_sample_counts.csv")
display(counts)
tradable = games[games["tradable"]]
print(f"Tradable games run from {tradable['game_date'].min()} to {tradable['game_date'].max()}")

,group,games
0,main sample games,1229
1,eligible (both teams have 10+ earlier games),1074
2,first 50 eligible games (threshold only),50
3,tradable games,1024


Tradable games run from 2025-11-17 to 2026-04-12


1,074 of the 1,229 games are eligible. The first 50 of them only set the threshold. That leaves 1,024 tradable games, from November 17, 2025 to April 12, 2026. All three strategies below are evaluated on these 1,024 games.

## 4. Results

Three strategies on the same tradable games:

- `volume_rule`: the rule above.
- `volume_benchmark`: buy the less popular team in every tradable game, with no threshold. This shows what the filter adds.
- `metro_rule`: the same rule with the metro population gap in place of the volume gap. The team from the smaller metro area is bought.

Return per dollar staked is profit divided by total cost plus total fees. The intervals are 95% percentile intervals from the block bootstrap by game date. The trades stay fixed in the bootstrap.

In [4]:
runs = [run_strategy(games, "volume_rule", "gap_volume", use_threshold=True),
        run_strategy(games, "volume_benchmark", "gap_volume", use_threshold=False),
        run_strategy(games, "metro_rule", "gap_metro", use_threshold=True)]
results = pd.DataFrame([row for _, row in runs])
trades = pd.concat([table[table["status"] != "no_signal"] for table, _ in runs],
                   ignore_index=True)
write_csv(results, RESULTS_DIR / "strategy_results.csv")
write_csv(trades, RESULTS_DIR / "strategy_trades.csv")
display(results.set_index("strategy").T.round(4))

strategy,volume_rule,volume_benchmark,metro_rule
tradable_games,1024.0000,1024.0000,1024.0000
signals,223.0000,1024.0000,275.0000
no_trade_bad_ask,0.0000,0.0000,0.0000
no_trade_gap_zero,0.0000,0.0000,0.0000
n_trades,223.0000,1024.0000,275.0000
wins,62.0000,379.0000,124.0000
win_rate,0.2780,0.3701,0.4509
mean_ask,0.3158,0.3910,0.4640
contracts,108240.0000,391910.0000,87386.0000
total_cost,22272.9700,102220.5100,27439.6400


**The rule (`volume_rule`).** It traded 223 of the 1,024 tradable games, or 21.8%. The share is under one quarter because the threshold moves during the season. Every signal had a usable ask, so no trade was skipped.

- The teams it bought won 62 of 223 games, a win rate of 27.8%. The average ask paid was 0.316. A contract bought at price P has to win more than P of the time to make money before fees, and here the win rate is below the average price.
- Total cost was $22,272.97 and total fees were $1,067.75. Without the cent rounding the fees would be $1,066.94.
- Profit: -$4,956.72. Return per dollar staked: -21.2%.
- Bootstrap intervals: profit from -$9,415 to +$235, return from -39.7% to +1.0%. Both intervals contain 0, by a small margin, so the loss is not distinguishable from zero at the 95% level. Nothing here points to a profit.
- Fees are $1,068 of the $4,957 loss. Before fees, the trades still lost $3,889.

**The benchmark (`volume_benchmark`).** Buying the less popular team in all 1,024 tradable games gave a win rate of 37.0% against an average ask of 0.391. Cost $102,220.51, fees $4,363.40, profit -$12,375.91, return -11.6%. The intervals exclude 0: profit from -$22,261 to -$2,656, return from -20.5% to -2.5%.

So buying the less popular team lost money across all tradable games, and the filter did not help: the rule's return (-21.2%) is lower than the benchmark's (-11.6%). The rule's interval is wider because it has about a fifth as many trades.

**Robustness (`metro_rule`).** With the metro gap, the rule traded 275 games (26.9%). Win rate 45.1% against an average ask of 0.464. Cost $27,439.64, fees $1,031.22, profit -$3,753.86, return -13.2%. The intervals exclude 0: profit from -$7,421 to -$146, return from -26.1% to -0.5%. The direction is the same as for the volume rule: a loss.

**How this fits with notebook 03.** There, at the same price, the team with more volume won more often than its price implied. A rule that buys the team with less volume is on the losing side of that pattern. It also pays the ask and the fee on every trade.

**What the numbers show.** Buying the less popular team at tipoff did not make money after fees in this season, with either measure of popularity or with no filter at all. The rule lost about 21 cents per dollar staked, with an interval from a 40-cent loss to a 1-cent gain.

**Limits of this backtest.** It covers one season. It assumes the whole order fills at the closing ask of the last minute before tipoff. That may not hold for orders of several hundred contracts, and the largest order here is 3,333 contracts. The quote used can also be carried forward from an earlier minute.

## 5. The trades

One row per order, for all three strategies. This is `results/strategy_trades.csv`.

In [5]:
display(trades[trades["strategy"] == "volume_rule"].drop(columns=["tipoff_utc"]).round(4))

,strategy,game_id,game_date,home_team,away_team,gap,threshold,signal,side,team,ask,status,contracts,cost,fee,fee_unrounded,won,payout,profit
0,volume_rule,0022500247,2025-11-17,NOP,OKC,-1.1452,0.6530,1,home,NOP,0.07,trade,1428,99.96,6.51,6.5074,0,0.0,-106.47
1,volume_rule,0022500248,2025-11-17,DEN,CHI,0.7129,0.6593,1,away,CHI,0.13,trade,769,99.97,6.09,6.0882,1,769.0,662.94
2,volume_rule,0022500249,2025-11-18,ORL,GSW,-0.9137,0.6774,1,home,ORL,0.43,trade,232,99.76,3.99,3.9804,1,232.0,128.25
3,volume_rule,0022500252,2025-11-18,SAS,MEM,0.7617,0.6892,1,away,MEM,0.33,trade,303,99.99,4.69,4.6895,0,0.0,-104.68
4,volume_rule,0022500253,2025-11-18,LAL,UTA,0.8925,0.7143,1,away,UTA,0.16,trade,625,100.00,5.88,5.8800,0,0.0,-105.88
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
218,volume_rule,0022501175,2026-04-10,IND,PHI,-0.5791,0.5728,1,home,IND,0.12,trade,833,99.96,6.16,6.1575,0,0.0,-106.12
219,volume_rule,0022501180,2026-04-10,SAS,DAL,0.9711,0.5740,1,away,DAL,0.07,trade,1428,99.96,6.51,6.5074,0,0.0,-106.47
220,volume_rule,0022501185,2026-04-10,LAL,PHX,0.8823,0.5731,1,away,PHX,0.56,trade,178,99.68,3.08,3.0701,0,0.0,-102.76
221,volume_rule,0022501188,2026-04-12,IND,DET,-0.6937,0.5739,1,home,IND,0.10,trade,1000,100.00,6.30,6.3000,0,0.0,-106.30


The table shows the first and last of the rule's 223 orders. Each order costs just under $100, because the number of contracts is rounded down.

Cheap contracts mean more contracts and a larger fee. The orders run from 105 contracts (ask 0.95, fee $0.35) to 3,333 contracts (ask 0.03, fee $6.79). The rule bought New Orleans most often (36 orders), then Memphis and Brooklyn (21 each).

Most orders lose the whole stake, which is expected when the average ask is 0.316. A few win several times the stake.

---

## Exploratory (not in the Stage 9 plan): profit and return before fees

**This section was added after the Stage 9 results above were seen.** It follows `ANALYSIS_PLAN.md`, "Exploratory, added after Stage 9 results", check E2, which was committed before this was run.

**Reason.** To show how much of each strategy's loss is fees.

The trades are the ones in section 4. No new rule is run. Profit before fees is total payout minus total cost, which is the profit plus the fees. Return before fees is that divided by total cost.

In [6]:
from strategy import before_fees_table

before_fees = before_fees_table(results)
write_csv(before_fees, RESULTS_DIR / "exploratory_strategy_before_fees.csv")
display(before_fees.set_index("strategy").T.round(4))

strategy,volume_rule,volume_benchmark,metro_rule
n_trades,223.0000,1024.0000,275.0000
total_cost,22272.9700,102220.5100,27439.6400
total_fees,1067.7500,4363.4000,1031.2200
profit,-4956.7200,-12375.9100,-3753.8600
return_per_dollar,-0.2124,-0.1161,-0.1318
profit_before_fees,-3888.9700,-8012.5100,-2722.6400
return_before_fees,-0.1746,-0.0784,-0.0992


All three strategies lose money before fees as well.

- **`volume_rule`.** Before fees the loss is $3,888.97, a return of -17.5% on the $22,272.97 spent. Fees of $1,067.75 bring it to the -$4,956.72 and -21.2% of section 4.
- **`volume_benchmark`.** Before fees the loss is $8,012.51, a return of -7.8%. Fees of $4,363.40 bring it to -$12,375.91 and -11.6%.
- **`metro_rule`.** Before fees the loss is $2,722.64, a return of -9.9%. Fees of $1,031.22 bring it to -$3,753.86 and -13.2%.

Fees are about a fifth of the volume rule's loss (22%), about a third of the benchmark's (35%) and about a quarter of the metro rule's (27%). The rest is the trades themselves: the teams bought won less often than the prices paid.

These figures have no bootstrap intervals. The intervals in section 4 are for the results after fees.